<a href="https://colab.research.google.com/github/huiningjiao02-ship-it/Huining-JIAO/blob/main/EEG_Emotion_Evaluation_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =========================
# 1. Install dependencies
# =========================
!pip install -q kagglehub torch torchvision torchaudio scikit-learn pandas numpy

In [ ]:
# =========================
# 2. Auto-download dataset
# =========================
# 不需要手动上传文件。
# 这个代码会自动下载 Kaggle: EEG Brainwave Dataset: Feeling Emotions
# 然后自动找到包含 label 列的 CSV 文件。

import os
import shutil
from pathlib import Path

import pandas as pd
import kagglehub

DATA_DIR = Path("/content/data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Downloading dataset from KaggleHub...")
dataset_path = kagglehub.dataset_download("birdy654/eeg-brainwave-dataset-feeling-emotions")
dataset_path = Path(dataset_path)

print("Dataset downloaded to:", dataset_path)

candidate_csvs = list(dataset_path.rglob("*.csv"))
print("CSV files found:")
for p in candidate_csvs[:20]:
    print(" -", p)

target_file = None

for csv_path in candidate_csvs:
    try:
        preview = pd.read_csv(csv_path, nrows=5)
        if "label" in preview.columns:
            target_file = csv_path
            break
    except Exception:
        pass

if target_file is None:
    raise FileNotFoundError(
        "No CSV file with a 'label' column was found. "
        "Please check the downloaded dataset structure."
    )

shutil.copy(target_file, DATA_DIR / "emotions.csv")

print("\\nSelected file:", target_file)
print("Saved as:", DATA_DIR / "emotions.csv")

df_preview = pd.read_csv(DATA_DIR / "emotions.csv", nrows=5)
print("\\nPreview:")
display(df_preview.head())

print("\\nShape:", pd.read_csv(DATA_DIR / "emotions.csv").shape)

100%|██████████| 11.9M/11.9M [00:00<00:00, 98.4MB/s]

Extracting files...


Dataset downloaded to: /root/.cache/kagglehub/datasets/birdy654/eeg-brainwave-dataset-feeling-emotions/versions/1
CSV files found:
 - /root/.cache/kagglehub/datasets/birdy654/eeg-brainwave-dataset-feeling-emotions/versions/1/emotions.csv
\nSelected file: /root/.cache/kagglehub/datasets/birdy654/eeg-brainwave-dataset-feeling-emotions/versions/1/emotions.csv
Saved as: /content/data/emotions.csv
\nPreview:


,# mean_0_a,mean_1_a,mean_2_a,mean_3_a,mean_4_a,mean_d_0_a,mean_d_1_a,mean_d_2_a,mean_d_3_a,mean_d_4_a,...,fft_741_b,fft_742_b,fft_743_b,fft_744_b,fft_745_b,fft_746_b,fft_747_b,fft_748_b,fft_749_b,label
0,4.62,30.3,-356.0,15.6,26.3,1.070,0.411,-15.70,2.06,3.15,...,23.5,20.3,20.3,23.5,-215.0,280.00,-162.00,-162.00,280.00,NEGATIVE
1,28.80,33.1,32.0,25.8,22.8,6.550,1.680,2.88,3.83,-4.82,...,-23.3,-21.8,-21.8,-23.3,182.0,2.57,-31.60,-31.60,2.57,NEUTRAL
2,8.90,29.4,-416.0,16.7,23.7,79.900,3.360,90.20,89.90,2.03,...,462.0,-233.0,-233.0,462.0,-267.0,281.00,-148.00,-148.00,281.00,POSITIVE
3,14.90,31.6,-143.0,19.8,24.3,-0.584,-0.284,8.82,2.30,-1.97,...,299.0,-243.0,-243.0,299.0,132.0,-12.40,9.53,9.53,-12.40,POSITIVE
4,28.30,31.3,45.2,27.3,24.5,34.800,-5.790,3.06,41.40,5.52,...,12.0,38.1,38.1,12.0,119.0,-17.60,23.90,23.90,-17.60,NEUTRAL


\nShape: (2132, 2549)


In [ ]:
# ============================================================
# 3. Main pipeline code
# ============================================================

import os
import json
import random
from pathlib import Path
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit, train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset


@dataclass
class ExperimentConfig:
    data_path: str = "/content/data/emotions.csv"
    output_dir: str = "/content/outputs"
    seed: int = 42

    # protocol: "holdout", "repeated_holdout", or "kfold"
    protocol: str = "kfold"
    test_size: float = 0.2
    n_splits: int = 5
    n_repeats: int = 5

    # model: "dummy", "logreg", or "cnn"
    model: str = "cnn"

    # CNN training
    epochs: int = 40
    batch_size: int = 32
    learning_rate: float = 1e-3
    weight_decay: float = 1e-2
    dropout: float = 0.5

    # diagnostic
    permutation_sanity_check: bool = False


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def load_feeling_emotions_csv(data_path: str):
    df = pd.read_csv(data_path)

    if "label" not in df.columns:
        raise ValueError("The CSV must contain a column named 'label'.")

    X = df.drop(columns=["label"]).values.astype(np.float32)
    y_raw = df["label"].values

    label_encoder = LabelEncoder()
    y = label_encoder.fit_transform(y_raw).astype(np.int64)
    class_names = list(label_encoder.classes_)

    print(f"Samples: {X.shape[0]}")
    print(f"Features: {X.shape[1]}")
    print(f"Classes: {class_names}")

    return X, y, class_names


def leakage_safe_standardize(X_train_raw, X_test_raw):
    """
    关键修改：
    只能在训练集上 fit StandardScaler，
    不能在全部数据上 fit_transform。
    """
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train_raw)
    X_test = scaler.transform(X_test_raw)

    return X_train.astype(np.float32), X_test.astype(np.float32)


class EEG1DCNN(nn.Module):
    """
    Lightweight 1D-CNN baseline.

    注意：
    Kaggle Feeling Emotions 数据集提供的是 EEG feature vector，
    不是原始多通道 EEG 时间序列。
    所以这里的 CNN 应该被描述为 compact neural baseline，
    不要夸大成完整的 spatio-temporal EEG model。
    """

    def __init__(self, input_dim: int, num_classes: int, dropout: float = 0.5):
        super().__init__()

        self.feature_extractor = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=5, padding=2),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(32),
        )

        with torch.no_grad():
            dummy = torch.zeros(1, 1, input_dim)
            out = self.feature_extractor(dummy)
            fc_dim = out.reshape(1, -1).shape[1]

        self.classifier = nn.Sequential(
            nn.Linear(fc_dim, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.feature_extractor(x)
        x = x.reshape(x.size(0), -1)
        return self.classifier(x)


def train_cnn_one_split(X_train, y_train, X_test, cfg: ExperimentConfig, num_classes: int):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    X_train_t = torch.tensor(X_train).unsqueeze(1)
    y_train_t = torch.tensor(y_train)
    X_test_t = torch.tensor(X_test).unsqueeze(1)

    train_loader = DataLoader(
        TensorDataset(X_train_t, y_train_t),
        batch_size=cfg.batch_size,
        shuffle=True,
    )

    model = EEG1DCNN(
        input_dim=X_train.shape[1],
        num_classes=num_classes,
        dropout=cfg.dropout,
    ).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=cfg.learning_rate,
        weight_decay=cfg.weight_decay,
    )
    criterion = nn.CrossEntropyLoss()

    best_state = None
    best_loss = float("inf")
    patience = 8
    no_improve = 0

    for epoch in range(cfg.epochs):
        model.train()
        total_loss = 0.0

        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * xb.size(0)

        avg_loss = total_loss / len(train_loader.dataset)

        if avg_loss < best_loss:
            best_loss = avg_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1

        if no_improve >= patience:
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    model.eval()
    preds = []

    test_loader = DataLoader(TensorDataset(X_test_t), batch_size=cfg.batch_size, shuffle=False)

    with torch.no_grad():
        for (xb,) in test_loader:
            xb = xb.to(device)
            logits = model(xb)
            pred = torch.argmax(logits, dim=1)
            preds.extend(pred.cpu().numpy())

    return np.array(preds)


def train_classical_one_split(X_train, y_train, X_test, cfg: ExperimentConfig):
    if cfg.model == "dummy":
        clf = DummyClassifier(strategy="most_frequent")

    elif cfg.model == "logreg":
        clf = LogisticRegression(
            max_iter=3000,
            class_weight="balanced",
            n_jobs=-1,
        )

    else:
        raise ValueError(f"Unsupported classical model: {cfg.model}")

    clf.fit(X_train, y_train)
    return clf.predict(X_test)


def evaluate_one_split(X, y, train_idx, test_idx, class_names, cfg: ExperimentConfig, split_name: str):
    X_train_raw = X[train_idx]
    X_test_raw = X[test_idx]
    y_train = y[train_idx].copy()
    y_test = y[test_idx].copy()

    if cfg.permutation_sanity_check:
        rng = np.random.default_rng(cfg.seed)
        y_train = rng.permutation(y_train)

    X_train, X_test = leakage_safe_standardize(X_train_raw, X_test_raw)

    if cfg.model == "cnn":
        y_pred = train_cnn_one_split(X_train, y_train, X_test, cfg, num_classes=len(class_names))
    else:
        y_pred = train_classical_one_split(X_train, y_train, X_test, cfg)

    acc = accuracy_score(y_test, y_pred)
    bal_acc = balanced_accuracy_score(y_test, y_pred)
    macro_f1 = f1_score(y_test, y_pred, average="macro")
    weighted_f1 = f1_score(y_test, y_pred, average="weighted")
    cm = confusion_matrix(y_test, y_pred)

    result = {
        "split": split_name,
        "model": cfg.model,
        "protocol": cfg.protocol,
        "accuracy": float(acc),
        "balanced_accuracy": float(bal_acc),
        "macro_f1": float(macro_f1),
        "weighted_f1": float(weighted_f1),
        "n_train": int(len(train_idx)),
        "n_test": int(len(test_idx)),
        "permutation_sanity_check": bool(cfg.permutation_sanity_check),
        "confusion_matrix": cm.tolist(),
        "classification_report": classification_report(
            y_test,
            y_pred,
            target_names=class_names,
            output_dict=True,
            zero_division=0,
        ),
    }

    print(
        f"{split_name:>10s} | "
        f"model={cfg.model:<6s} | "
        f"acc={acc:.4f} | "
        f"bal_acc={bal_acc:.4f} | "
        f"macro_f1={macro_f1:.4f} | "
        f"weighted_f1={weighted_f1:.4f}"
    )

    return result


def run_protocol(X, y, class_names, cfg: ExperimentConfig):
    results = []

    if cfg.protocol == "holdout":
        train_idx, test_idx = train_test_split(
            np.arange(len(y)),
            test_size=cfg.test_size,
            random_state=cfg.seed,
            stratify=y,
        )
        results.append(evaluate_one_split(X, y, train_idx, test_idx, class_names, cfg, "holdout"))

    elif cfg.protocol == "repeated_holdout":
        splitter = StratifiedShuffleSplit(
            n_splits=cfg.n_repeats,
            test_size=cfg.test_size,
            random_state=cfg.seed,
        )

        for i, (train_idx, test_idx) in enumerate(splitter.split(X, y), start=1):
            results.append(evaluate_one_split(X, y, train_idx, test_idx, class_names, cfg, f"repeat_{i}"))

    elif cfg.protocol == "kfold":
        splitter = StratifiedKFold(
            n_splits=cfg.n_splits,
            shuffle=True,
            random_state=cfg.seed,
        )

        for i, (train_idx, test_idx) in enumerate(splitter.split(X, y), start=1):
            results.append(evaluate_one_split(X, y, train_idx, test_idx, class_names, cfg, f"fold_{i}"))

    else:
        raise ValueError("protocol must be one of: holdout, repeated_holdout, kfold")

    return results


def summarize_results(results):
    df = pd.DataFrame([
        {
            "split": r["split"],
            "model": r["model"],
            "protocol": r["protocol"],
            "accuracy": r["accuracy"],
            "balanced_accuracy": r["balanced_accuracy"],
            "macro_f1": r["macro_f1"],
            "weighted_f1": r["weighted_f1"],
            "n_train": r["n_train"],
            "n_test": r["n_test"],
            "permutation_sanity_check": r["permutation_sanity_check"],
        }
        for r in results
    ])

    print("\n===== Summary: mean ± std =====")
    for metric in ["accuracy", "balanced_accuracy", "macro_f1", "weighted_f1"]:
        mean = df[metric].mean()
        std = df[metric].std(ddof=1) if len(df) > 1 else 0.0
        print(f"{metric:>18s}: {mean:.4f} ± {std:.4f}")

    return df


def run_experiment(
    model="cnn",
    protocol="kfold",
    data_path="/content/data/emotions.csv",
    output_dir="/content/outputs",
    seed=42,
    n_splits=5,
    n_repeats=5,
    epochs=40,
    permutation_sanity_check=False,
):
    cfg = ExperimentConfig(
        data_path=data_path,
        output_dir=output_dir,
        seed=seed,
        model=model,
        protocol=protocol,
        n_splits=n_splits,
        n_repeats=n_repeats,
        epochs=epochs,
        permutation_sanity_check=permutation_sanity_check,
    )

    set_seed(cfg.seed)
    Path(cfg.output_dir).mkdir(parents=True, exist_ok=True)

    X, y, class_names = load_feeling_emotions_csv(cfg.data_path)

    print("\n===== Running experiment =====")
    print(asdict(cfg))
    print()

    results = run_protocol(X, y, class_names, cfg)
    summary_df = summarize_results(results)

    tag = f"{cfg.model}_{cfg.protocol}"
    if cfg.permutation_sanity_check:
        tag += "_permutation"

    summary_path = Path(cfg.output_dir) / f"summary_{tag}.csv"
    details_path = Path(cfg.output_dir) / f"details_{tag}.json"
    config_path = Path(cfg.output_dir) / f"config_{tag}.json"

    summary_df.to_csv(summary_path, index=False)

    with open(details_path, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)

    with open(config_path, "w", encoding="utf-8") as f:
        json.dump(asdict(cfg), f, indent=2, ensure_ascii=False)

    print("\nSaved files:")
    print(summary_path)
    print(details_path)
    print(config_path)

    return summary_df, results

In [ ]:
# ============================================================
# 4. Run baseline benchmarking
# ============================================================
# 建议顺序：先跑 dummy 和 logreg，再跑 cnn。

summary_dummy, results_dummy = run_experiment(
    model="dummy",
    protocol="kfold",
    epochs=1,
)

summary_logreg, results_logreg = run_experiment(
    model="logreg",
    protocol="kfold",
    epochs=1,
)

summary_cnn, results_cnn = run_experiment(
    model="cnn",
    protocol="kfold",
    epochs=40,
)

Samples: 2132
Features: 2548
Classes: ['NEGATIVE', 'NEUTRAL', 'POSITIVE']

===== Running experiment =====
{'data_path': '/content/data/emotions.csv', 'output_dir': '/content/outputs', 'seed': 42, 'protocol': 'kfold', 'test_size': 0.2, 'n_splits': 5, 'n_repeats': 5, 'model': 'dummy', 'epochs': 1, 'batch_size': 32, 'learning_rate': 0.001, 'weight_decay': 0.01, 'dropout': 0.5, 'permutation_sanity_check': False}

    fold_1 | model=dummy  | acc=0.3349 | bal_acc=0.3333 | macro_f1=0.1673 | weighted_f1=0.1680
    fold_2 | model=dummy  | acc=0.3349 | bal_acc=0.3333 | macro_f1=0.1673 | weighted_f1=0.1680
    fold_3 | model=dummy  | acc=0.3357 | bal_acc=0.3333 | macro_f1=0.1675 | weighted_f1=0.1687
    fold_4 | model=dummy  | acc=0.3380 | bal_acc=0.3333 | macro_f1=0.1684 | weighted_f1=0.1708
    fold_5 | model=dummy  | acc=0.3357 | bal_acc=0.3333 | macro_f1=0.1675 | weighted_f1=0.1687

===== Summary: mean ± std =====
          accuracy: 0.3358 ± 0.0013
 balanced_accuracy: 0.3333 ± 0.0000
       

In [ ]:
# ============================================================
# 5. Optional: label permutation sanity check
# ============================================================
# 这个实验用于检查模型是否可能依赖泄漏或伪相关。
# 如果标签打乱后结果仍然很高，就说明实验非常可疑。

summary_perm, results_perm = run_experiment(
    model="cnn",
    protocol="kfold",
    epochs=20,
    permutation_sanity_check=True,
)

Samples: 2132
Features: 2548
Classes: ['NEGATIVE', 'NEUTRAL', 'POSITIVE']

===== Running experiment =====
{'data_path': '/content/data/emotions.csv', 'output_dir': '/content/outputs', 'seed': 42, 'protocol': 'kfold', 'test_size': 0.2, 'n_splits': 5, 'n_repeats': 5, 'model': 'cnn', 'epochs': 20, 'batch_size': 32, 'learning_rate': 0.001, 'weight_decay': 0.01, 'dropout': 0.5, 'permutation_sanity_check': True}

    fold_1 | model=cnn    | acc=0.2763 | bal_acc=0.2770 | macro_f1=0.1711 | weighted_f1=0.1707
    fold_2 | model=cnn    | acc=0.4637 | bal_acc=0.4627 | macro_f1=0.3939 | weighted_f1=0.3943
    fold_3 | model=cnn    | acc=0.3192 | bal_acc=0.3215 | macro_f1=0.1817 | weighted_f1=0.1804
    fold_4 | model=cnn    | acc=0.3545 | bal_acc=0.3501 | macro_f1=0.2216 | weighted_f1=0.2236
    fold_5 | model=cnn    | acc=0.3685 | bal_acc=0.3664 | macro_f1=0.2321 | weighted_f1=0.2329

===== Summary: mean ± std =====
          accuracy: 0.3565 ± 0.0697
 balanced_accuracy: 0.3555 ± 0.0688
         

In [ ]:
# ============================================================
# 6. Show output files
# ============================================================

import os

for root, dirs, files in os.walk("/content/outputs"):
    for f in files:
        print(os.path.join(root, f))

/content/outputs/details_cnn_kfold.json
/content/outputs/summary_dummy_kfold.csv
/content/outputs/summary_cnn_kfold.csv
/content/outputs/config_cnn_kfold_permutation.json
/content/outputs/summary_cnn_kfold_permutation.csv
/content/outputs/config_cnn_kfold.json
/content/outputs/config_logreg_kfold.json
/content/outputs/details_cnn_kfold_permutation.json
/content/outputs/details_logreg_kfold.json
/content/outputs/details_dummy_kfold.json
/content/outputs/summary_logreg_kfold.csv
/content/outputs/config_dummy_kfold.json


In [ ]:
# ============================================================
# 7. Download outputs
# ============================================================

!zip -r /content/eeg_emotion_outputs.zip /content/outputs

from google.colab import files
files.download("/content/eeg_emotion_outputs.zip")

  adding: content/outputs/ (stored 0%)
  adding: content/outputs/details_cnn_kfold.json (deflated 83%)
  adding: content/outputs/summary_dummy_kfold.csv (deflated 63%)
  adding: content/outputs/summary_cnn_kfold.csv (deflated 53%)
  adding: content/outputs/config_cnn_kfold_permutation.json (deflated 41%)
  adding: content/outputs/summary_cnn_kfold_permutation.csv (deflated 48%)
  adding: content/outputs/config_cnn_kfold.json (deflated 41%)
  adding: content/outputs/config_logreg_kfold.json (deflated 41%)
  adding: content/outputs/details_cnn_kfold_permutation.json (deflated 81%)
  adding: content/outputs/details_logreg_kfold.json (deflated 83%)
  adding: content/outputs/details_dummy_kfold.json (deflated 89%)
  adding: content/outputs/summary_logreg_kfold.csv (deflated 53%)
  adding: content/outputs/config_dummy_kfold.json (deflated 41%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>